# Building Comparison: 2016 vs 2025
### Using Google Earth Engine to detect urban development

This notebook compares satellite imagery to identify new buildings and urban expansion between 2016 and 2025.

## Setup and Installation
First time setup - run these commands in terminal:
```bash
pip install earthengine-api geemap
earthengine authenticate
```

In [1]:
import ee
import geemap

# REPLACE THIS with the Personal ID you just found/created
my_personal_project = 'sensiblesat' 

try:
    ee.Initialize(project=my_personal_project)
    print(f"Success! Connected to personal project: {my_personal_project}")
except Exception as e:
    print("New auth needed for this project...")
    # This might pop up the browser one last time to link THIS project
    ee.Authenticate()
    ee.Initialize(project=my_personal_project)

Success! Connected to personal project: sensiblesat


## Define Area of Interest
Choose your location to analyze. Examples provided below:

In [2]:
# Choose one of these locations or define your own:

# San Francisco Bay Area
aoi = ee.Geometry.Rectangle([-122.5, 37.7, -122.3, 37.9])
location_name = "San Francisco"

# Dubai (rapid development)
# aoi = ee.Geometry.Rectangle([55.1, 25.0, 55.5, 25.4])
# location_name = "Dubai"

# Beijing
# aoi = ee.Geometry.Rectangle([116.2, 39.8, 116.6, 40.1])
# location_name = "Beijing"

# Las Vegas
# aoi = ee.Geometry.Rectangle([-115.3, 36.0, -115.0, 36.3])
# location_name = "Las Vegas"

# Or create from coordinates with buffer:
# point = ee.Geometry.Point([longitude, latitude])
# aoi = point.buffer(5000)  # 5km radius
# location_name = "Custom Location"

print(f"Analyzing: {location_name}")

Analyzing: San Francisco


## Define Helper Functions

In [3]:
def get_yearly_composite(year, aoi):
    """Get cloud-free Sentinel-2 composite for a specific year"""
    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year, 12, 31)
    
    composite = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .median()
        .clip(aoi))
    
    return composite

def calculate_ndbi(image):
    """Calculate Normalized Difference Built-up Index
    NDBI = (SWIR - NIR) / (SWIR + NIR)
    Higher values indicate built-up areas
    """
    return image.normalizedDifference(['B11', 'B8']).rename('NDBI')

def calculate_ndvi(image):
    """Calculate Normalized Difference Vegetation Index"""
    return image.normalizedDifference(['B8', 'B4']).rename('NDVI')

def get_area_stats(image, geometry):
    """Calculate area in square meters"""
    area = image.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geometry,
        scale=10,
        maxPixels=1e9
    )
    return area

print("✓ Functions defined")

✓ Functions defined


## Load and Process Satellite Imagery

In [4]:
print("Loading satellite imagery...")

# Get imagery for both years
image_2016 = get_yearly_composite(2016, aoi)
image_2025 = get_yearly_composite(2025, aoi)

print("✓ Imagery loaded")

# Calculate building indices
print("Calculating building indices...")
ndbi_2016 = calculate_ndbi(image_2016)
ndbi_2025 = calculate_ndbi(image_2025)

ndvi_2016 = calculate_ndvi(image_2016)
ndvi_2025 = calculate_ndvi(image_2025)

print("✓ Indices calculated")

Loading satellite imagery...
✓ Imagery loaded
Calculating building indices...
✓ Indices calculated


## Detect Buildings and Changes

In [5]:
# Building detection threshold
# Adjust this value based on your area (typically 0.0 to 0.2)
# Higher values = more strict detection (fewer buildings)
# Lower values = more lenient detection (more buildings)
building_threshold = 0.1

# Create binary masks for buildings
buildings_2016 = ndbi_2016.gt(building_threshold).selfMask()
buildings_2025 = ndbi_2025.gt(building_threshold).selfMask()

# Detect changes
new_buildings = (ndbi_2025.gt(building_threshold)
    .And(ndbi_2016.lt(building_threshold))
    .selfMask())

lost_buildings = (ndbi_2016.gt(building_threshold)
    .And(ndbi_2025.lt(building_threshold))
    .selfMask())

print("✓ Building detection complete")

✓ Building detection complete


## Calculate Statistics

In [6]:
print("Calculating statistics...")

stats_2016 = get_area_stats(buildings_2016, aoi)
stats_2025 = get_area_stats(buildings_2025, aoi)
new_building_stats = get_area_stats(new_buildings, aoi)
lost_building_stats = get_area_stats(lost_buildings, aoi)

# Retrieve and display statistics
area_2016 = stats_2016.getInfo().get('NDBI', 0)
area_2025 = stats_2025.getInfo().get('NDBI', 0)
new_area = new_building_stats.getInfo().get('NDBI', 0)
lost_area = lost_building_stats.getInfo().get('NDBI', 0)

print(f"\n{'='*60}")
print(f"BUILDING CHANGE ANALYSIS: {location_name}")
print(f"Period: 2016 → 2025")
print(f"{'='*60}")
print(f"Built-up area 2016:  {area_2016/1e6:.2f} km²")
print(f"Built-up area 2025:  {area_2025/1e6:.2f} km²")
print(f"New construction:    {new_area/1e6:.2f} km²")
print(f"Lost/demolished:     {lost_area/1e6:.2f} km²")
print(f"Net change:          {(area_2025-area_2016)/1e6:.2f} km²")
if area_2016 > 0:
    print(f"Percent increase:    {(area_2025-area_2016)/area_2016*100:.1f}%")
print(f"{'='*60}\n")

Calculating statistics...

BUILDING CHANGE ANALYSIS: San Francisco
Period: 2016 → 2025
Built-up area 2016:  38.62 km²
Built-up area 2025:  33.69 km²
New construction:    12.59 km²
Lost/demolished:     17.52 km²
Net change:          -4.94 km²
Percent increase:    -12.8%



## Visualize Results

In [7]:
# Visualization parameters
rgb_vis = {
    'min': 0,
    'max': 3000,
    'bands': ['B4', 'B3', 'B2']
}

ndbi_vis = {
    'min': -0.5,
    'max': 0.5,
    'palette': ['blue', 'white', 'red']
}

# Create interactive map
Map = geemap.Map()
Map.centerObject(aoi, 12)

# Add base imagery
Map.addLayer(image_2016, rgb_vis, '📅 2016 RGB', False)
Map.addLayer(image_2025, rgb_vis, '📅 2025 RGB', True)

# Add building indices
Map.addLayer(ndbi_2016, ndbi_vis, '📊 2016 NDBI', False)
Map.addLayer(ndbi_2025, ndbi_vis, '📊 2025 NDBI', False)

# Add building layers
Map.addLayer(buildings_2016, {'palette': ['yellow']}, '🏢 2016 Buildings', False)
Map.addLayer(buildings_2025, {'palette': ['orange']}, '🏢 2025 Buildings', False)

# Add change detection layers
Map.addLayer(new_buildings, {'palette': ['#00FF00']}, '🟢 New Buildings', True)
Map.addLayer(lost_buildings, {'palette': ['#FF0000']}, '🔴 Lost Buildings', False)

# Add area boundary
Map.addLayer(aoi, {'color': 'cyan'}, 'Area of Interest', False)

# Add controls
Map.add_layer_control()

# Add legend
legend_dict = {
    'New Buildings': '#00FF00',
    'Lost Buildings': '#FF0000',
    '2025 Buildings': 'orange',
    '2016 Buildings': 'yellow'
}
Map.add_legend(title="Legend", legend_dict=legend_dict)

# Display map
Map

Map(center=[37.799997134973765, -122.40000000000165], controls=(WidgetControl(options=['position', 'transparen…

## Create Split-Panel Comparison

In [8]:
# Create side-by-side comparison
left_layer = geemap.ee_tile_layer(image_2016, rgb_vis, '2016')
right_layer = geemap.ee_tile_layer(image_2025, rgb_vis, '2025')

split_map = geemap.Map()
split_map.centerObject(aoi, 12)
split_map.split_map(left_layer, right_layer)

split_map

Map(center=[37.799997134973765, -122.40000000000165], controls=(ZoomControl(options=['position', 'zoom_in_text…

## Export Results (Optional)
Export the change detection results as a GeoTIFF for use in other GIS software

In [ ]:
# Uncomment to export to Google Drive
# task = ee.batch.Export.image.toDrive(
#     image=new_buildings,
#     description='new_buildings_2016_2025',
#     folder='GEE_Exports',
#     region=aoi,
#     scale=10,
#     maxPixels=1e9
# )
# task.start()
# print("Export task started. Check Google Drive for results.")

## Tips and Notes

- **Green areas**: New buildings constructed between 2016-2025
- **Red areas**: Buildings demolished or changed to non-built land
- **Adjust threshold**: If you see too many/few buildings, modify `building_threshold`
- **Resolution**: Sentinel-2 has 10m resolution, so small buildings may not be detected
- **Cloud cover**: Some areas may have persistent clouds affecting accuracy
- **Urban areas work best**: This method is optimized for detecting built-up areas

### Recommended threshold values:
- Dense urban areas: 0.1 - 0.15
- Suburban areas: 0.05 - 0.1
- Rural areas: 0.0 - 0.05